<a href="https://colab.research.google.com/github/JeroCQ/Meeting-Summarizer/blob/just-transcript/meeting_to_context_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 25.1 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.10.0
    Uninstalling google-genai-2.10.0:
      Successfully uninstalled google-genai-2.10.0


In [26]:
import os
import time
from google.colab import auth
from googleapiclient.discovery import build
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from google.colab import userdata

# 1. Authenticate with Google Drive & Docs
auth.authenticate_user()
docs_service = build('docs', 'v1')
drive_service = build('drive', 'v3')

# 2. Setup New Gemini Client
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# TODO: Replace this with your actual Google Doc Template ID
TEMPLATE_DOCUMENT_ID = "1gbZPhUeQ5cw-T3tL51mvwZ0RoOSEz8F1Uoa27gBxPw0"
AUDIO_FILENAME = "meeting-july-11-2026.m4a"

# 3. Pydantic Schema optimized to capture raw, exhaustive, uncompressed context details
class MeetingSummary(BaseModel):
    document_title: str = Field(
        description="Extract the date from the provided filename and format it exactly as: meeting-to-context-window-yy-mm-dd"
    )
    add_to_context_window: str = Field(
        description="Exhaustive, granular, and hyper-detailed engineering-ready prompt context. "
                    "Must explicitly catalog every micro-detail discussed in the audio: exact product measurements "
                    "(cuajada, cream by pound/kilo/redondos), complete warehouse pickup protocols, "
                    "specific social media channels (TikTok, FB, IG), comprehensive banking/QR transaction steps, "
                    "and the four explicit Chatwoot human handoff triggers down to their exact phrasing constraints."
    )

# 4. Prompt optimized for maximum depth and absolute granularity
prompt = f"""
Task: Analyze this meeting audio and extract an exhaustive, uncompressed operational dataset to feed directly into a WhatsApp customer service chatbot's context window.

File Context: The audio you are analyzing is named '{AUDIO_FILENAME}'. Use this filename to determine the meeting date for the document title.

Focus Framework:
- Product Details: Extract all catalog parameters (cuajada, cream, portions, pricing by pound/kilo/redondos).
- Funnel Target: Parameters targeting new customers from Facebook, TikTok, and Instagram ads.
- Warehouse & Logistics: Every constraint regarding the physical warehouse/point of sale direct pickup workflow.
- Financial Validation: Exact bank transfer parameters (QR codes, company accounts, receipt upload mandates).
- Chatwoot Escalation Protocol: Deepest granular rules for human agent handoff:
  1. Wholesale intent detection.
  2. Manual payment verification requests.
  3. Sensitive inquiries (owner name, personal info, direct agent demands).
  4. Stalled or unresolved retail interactions.

Constraint: DO NOT summarize or truncate. Capture every minute detail, micro-decision, asset mention, and operational rule found in the audio transcript. Extend the text as long as necessary.
"""

# 5. Upload Audio via the New SDK
print(f"Uploading audio file: {AUDIO_FILENAME}...")
audio_file = client.files.upload(file=AUDIO_FILENAME)

# Wait for processing
while audio_file.state.name == 'PROCESSING':
    print('.', end='')
    time.sleep(5)
    audio_file = client.files.get(name=audio_file.name)

if audio_file.state.name == 'FAILED':
    raise ValueError("Audio processing failed.")
print("\nAudio file ready!")

# 6. Generate Content using Structured Output (gemini-flash-latest handles file parsing beautifully)
print("Generating exhaustive chatbot context from audio...")
response = client.models.generate_content(
    model='gemini-flash-latest',
    contents=[prompt, audio_file],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MeetingSummary,
        temperature=0.1
    )
)

summary_data = response.parsed
print("Success! Chatbot context parsed.")

# 7. Generate the Google Doc
print(f"Creating new Google Doc from template: {summary_data.document_title}...")
copied_file = drive_service.files().copy(
    fileId=TEMPLATE_DOCUMENT_ID,
    body={'name': summary_data.document_title}
).execute()
new_doc_id = copied_file.get('id')

# Build the batchUpdate requests array targeting the single placeholder string
requests = [
    {
        'replaceAllText': {
            'containsText': {
                'text': '{{ADD_TO_CONTEXT_WINDOW}}',
                'matchCase': True
            },
            'replaceText': summary_data.add_to_context_window
        }
    }
]

# Execute the batch update on the Google Docs API
docs_service.documents().batchUpdate(
    documentId=new_doc_id,
    body={'requests': requests}
).execute()

print(f"\nDocument generated successfully! Access your raw chatbot context text here:")
print(f"https://docs.google.com/document/d/{new_doc_id}/edit")

Uploading audio file: meeting-july-11-2026.m4a...

Audio file ready!
Generating exhaustive chatbot context from audio...


Success! Chatbot context parsed.
Creating new Google Doc from template: meeting-to-context-window-26-07-11...



Document generated successfully! Access your raw chatbot context text here:
https://docs.google.com/document/d/1LRqg2lKOXTiZVdKqf4srW0cJevmTd54m4oeVMu2ln9s/edit
